In [0]:
%run ../utilities/file_utilities


In [0]:
%run ../utilities/extract_join_keys

In [0]:
dbutils.widgets.text("table_name", "")


In [0]:
from pyspark.sql import SparkSession
from delta.tables import DeltaTable
from pyspark.sql.types import *

from pyspark.sql.functions import *
import json
import os
spark = SparkSession.builder.appName("fep_ben_mdrnz_apg_2_databricks").getOrCreate()



try:
    # Load Postgres connection parameters from JSON file
    with open('../parameter_config/unload_apg_metdata.json', 'r') as f:
        conn_params = json.load(f)
except json.JSONDecodeError as e:
        print(f"Error parsing parameter config JSON: {e}")
        raise
except FileNotFoundError:
        print(" parameter config File not found! Check your path.")
        raise
except Exception as e:
        print(f"Unexpected error: {e}")
        raise


table_name=dbutils.widgets.get("table_name")
conn_config=conn_params['conn_config']
target_catalog=conn_params['databricks_catalog_name']
join_key=conn_params['join_keys']
full_table_nm=f"{target_catalog}.silver.{table_name}"
checkpoint_file_path="/Volumes/benefit_modernization_dev/silver/delta_checkpoint_files"
#delta_checkpoint_file=f"abfss://benefit-modernization@enterprzmdrnzetl2025.dfs.core.windows.net/delta_checkpoint_files/{table_name}.checkpoint"
delta_checkpoint_file=f"{checkpoint_file_path}/{table_name}.checkpoint"




if(path_exists(delta_checkpoint_file)==False and not spark.catalog.tableExists(full_table_nm)):
         df=spark.read.table(f"{target_catalog}.bronze.{table_name}")

         df_hash_val=df.withColumn("hash_val", sha2(concat_ws("||",*[coalesce(col(c).cast("string"), lit("NULL")) for c in df.columns]),256)).withColumn("start_date",current_timestamp()).withColumn("end_date",to_timestamp(lit("9999-12-31 23:59:59.99"))).withColumn("current_flag",lit("Y"))

         df_hash_val.write.format("delta").mode("append").saveAsTable(full_table_nm)

         df.select(max(col("UPDATED_AT")).alias("max_updated_at")).write.format("csv").mode("overwrite").save(delta_checkpoint_file)

elif(path_exists(delta_checkpoint_file)==False and spark.catalog.tableExists(full_table_nm)):
        df_count=spark.read.table(full_table_nm).count()
        if(df_count==0):
                df=spark.read.table(f"{target_catalog}.bronze.{table_name}")

                df_hash_val=df.withColumn("hash_val", sha2(concat_ws("||",*[coalesce(col(c).cast("string"), lit("NULL")) for c in df.columns]),256)).withColumn("start_date",current_timestamp()).withColumn("end_date",to_timestamp(lit("9999-12-31 23:59:59.99"))).withColumn("current_flag",lit("Y"))

                df_hash_val.write.format("delta").mode("append").saveAsTable(full_table_nm)

                df.select(max(col("UPDATED_AT")).alias("max_updated_at")).write.format("csv").mode("overwrite").save(delta_checkpoint_file)
        else:
                df=spark.read.table(full_table_nm)
                
                df.select(max(col("UPDATED_AT")).alias("max_updated_at")).write.format("csv").mode("overwrite").save(delta_checkpoint_file)

                checkpoint_val=spark.read.csv(delta_checkpoint_file).take(1)[0][0];

                df_cdc_silver=spark.read.table(full_table_nm)

                df_cdc_bronze=spark.read.table(f"{target_catalog}.bronze.{table_name}").filter(col("CREATED_AT")>checkpoint_val ).filter(col("UPDATED_AT")>checkpoint_val)

                df_cdc_bronze_intmd=df_cdc_bronze.withColumn("hash_val", sha2(concat_ws("||",*[coalesce(col(c).cast("string"), lit("NULL")) for c in df_cdc_bronze.columns]),256))

                df_cdc_bronze_final=df_cdc_bronze_intmd.select(*[col(c).alias(f"bronze_{c}") for c in df_cdc_bronze_intmd.columns])

                join_expr=get_join_keys(table_name,join_key)
     
                df_cdc_insert=df_cdc_bronze_final.alias("l").join(df_cdc_silver.alias("r"),expr(join_expr),how="left_anti").select(*[col(c) for c in df_cdc_bronze_final.columns]).withColumn("start_date",current_timestamp()).withColumn("end_date",to_timestamp(lit("9999-12-31 23:59:59.99"))).withColumn("current_flag",lit("Y"))


                df_cdc_update=df_cdc_bronze_final.alias("l").join(df_cdc_silver.alias("r"), (expr(join_expr)) & (col('r.hash_val') != col('l.bronze_hash_val')), how="inner").select(*[col(c) for c in df_cdc_bronze_final.columns]).withColumn("start_date",current_timestamp()).withColumn("end_date",to_timestamp(lit("9999-12-31 23:59:59.99"))).withColumn("current_flag",lit("Y"))

                df_cdc_insert_final = df_cdc_insert.select(*[col(c).alias(c.replace("bronze_","",1)) for c in df_cdc_insert.columns])
                df_cdc_update_final = df_cdc_update.select(*[col(c).alias(c.replace("bronze_","",1)) for c in df_cdc_update.columns])

                df_cdc_insert_union=df_cdc_insert_final.unionByName(df_cdc_update_final).withColumn('operation',lit('I'))
                df_cdc_update_union=df_cdc_update_final.withColumn('operation',lit('U'))                                                                      

                df_scd2=df_cdc_insert_union.unionByName(df_cdc_update_union)

                df_cdc_silver.createOrReplaceTempView("target_table")
                df_scd2.createOrReplaceTempView("delta_table")

                merge_expr=get_merge_keys(table_name,join_key)
                merge_query='MERGE INTO target_table t'+' ' +'USING delta_table u'+" "+" ON "+merge_expr+" "+"WHEN MATCHED AND u.operation = 'U' THEN UPDATE SET"+" "+"t.current_flag = 'N',"+" "+"t.end_date = current_timestamp()"+" "+"WHEN NOT MATCHED and u.operation = 'I' THEN INSERT *"
                               
                spark.sql(merge_query)

                df_max_checkpoint=spark.read.table(full_table_nm)
                df_max_checkpoint.select(max(col("UPDATED_AT")).alias("max_updated_at")).write.format("csv").mode("overwrite").save(delta_checkpoint_file)

                

                
               


else:

        print("checkpoint file exists")
        df_count=spark.read.table(full_table_nm).count()
        if(df_count==0):
                df=spark.read.table(f"{target_catalog}.bronze.{table_name}")

                df_hash_val=df.withColumn("hash_val",sha2(concat_ws("||",*[coalesce(col(c).cast("string"), lit("NULL")) for c in df.columns]),256)).withColumn("start_date",current_timestamp()).withColumn("end_date",to_timestamp(lit("9999-12-31 23:59:59.99"))).withColumn("current_flag",lit("Y"))

                df_hash_val.write.format("delta").mode("append").saveAsTable(full_table_nm)
                
                df.select(max(col("CREATED_AT")).alias("max_created_at")).write.format("csv").mode("overwrite").save(delta_checkpoint_file)
        else:
                checkpoint_val=spark.read.csv(delta_checkpoint_file).take(1)[0][0];

                df_cdc_silver=spark.read.table(full_table_nm)

                df_cdc_bronze=spark.read.table(f"{target_catalog}.bronze.{table_name}").filter(col("CREATED_AT")>checkpoint_val ).filter(col("UPDATED_AT")>checkpoint_val)

                df_cdc_bronze_intmd=df_cdc_bronze.withColumn("hash_val", sha2(concat_ws("||",*[coalesce(col(c).cast("string"), lit("NULL")) for c in df_cdc_bronze.columns]),256))

                df_cdc_bronze_final=df_cdc_bronze_intmd.select(*[col(c).alias(f"bronze_{c}") for c in df_cdc_bronze_intmd.columns])

                join_expr=get_join_keys(table_name,join_key)
     
                df_cdc_insert=df_cdc_bronze_final.alias("l").join(df_cdc_silver.alias("r"),expr(join_expr),how="left_anti").select(*[col(c) for c in df_cdc_bronze_final.columns]).withColumn("start_date",current_timestamp()).withColumn("end_date",to_timestamp(lit("9999-12-31 23:59:59.99"))).withColumn("current_flag",lit("Y"))


                df_cdc_update=df_cdc_bronze_final.alias("l").join(df_cdc_silver.alias("r"), (expr(join_expr)) & (col('r.hash_val') != col('l.bronze_hash_val')), how="inner").select(*[col(c) for c in df_cdc_bronze_final.columns]).withColumn("start_date",current_timestamp()).withColumn("end_date",to_timestamp(lit("9999-12-31 23:59:59.99"))).withColumn("current_flag",lit("Y"))

                df_cdc_insert_final = df_cdc_insert.select(*[col(c).alias(c.replace("bronze_","",1)) for c in df_cdc_insert.columns])
                df_cdc_update_final = df_cdc_update.select(*[col(c).alias(c.replace("bronze_","",1)) for c in df_cdc_update.columns])

                df_cdc_insert_union=df_cdc_insert_final.unionByName(df_cdc_update_final).withColumn('operation',lit('I'))
                df_cdc_update_union=df_cdc_update_final.withColumn('operation',lit('U'))                                                                      

                df_scd2=df_cdc_insert_union.unionByName(df_cdc_update_union)

                df_cdc_silver.createOrReplaceTempView("target_table")
                df_scd2.createOrReplaceTempView("delta_table")

                merge_expr=get_merge_keys(table_name,join_key)
                merge_query='MERGE INTO target_table t'+' ' +'USING delta_table u'+" "+" ON "+merge_expr+" "+"WHEN MATCHED AND u.operation = 'U' THEN UPDATE SET"+" "+"t.current_flag = 'N',"+" "+"t.end_date = current_timestamp()"+" "+"WHEN NOT MATCHED and u.operation = 'I' THEN INSERT *"
                               
                spark.sql(merge_query)

                df_max_checkpoint=spark.read.table(full_table_nm)
                df_max_checkpoint.select(max(col("UPDATED_AT")).alias("max_updated_at")).write.format("csv").mode("overwrite").save(delta_checkpoint_file)



                


                

                
               
                

                                

                

                                                                      
                



                


                
               
                
                    

                
      


         
      